# Learning 02: NER & Classification Demos with Real Models

Building production-quality Gradio demos with:
- **GLiNER** (`knowledgator/gliner-bi-edge-v2.0`) — zero-shot NER
- **GLiClass** (`knowledgator/gliclass-edge-v3.0`) — zero-shot text classification

Key pattern: load model **once** outside Gradio callbacks to avoid reloading on every request.

In [ ]:
import json, os
import gradio as gr
from gliner import GLiNER
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "sample_texts.json")) as f:
    SAMPLE_TEXTS = json.load(f)

print(f"✓ Loaded {len(SAMPLE_TEXTS)} sample texts")

## 1. Load Models Once

In [ ]:
# GLiNER — zero-shot NER
ner_model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")
print("✓ GLiNER loaded")

# GLiClass — zero-shot classification
cls_model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
cls_tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)
cls_pipeline = ZeroShotClassificationPipeline(cls_model, cls_tokenizer, classification_type='multi-label', device='cpu')
print("✓ GLiClass loaded")

## 2. NER Demo with HighlightedText

`gr.HighlightedText` expects a list of `(text_chunk, label_or_None)` tuples.
We split the original text at entity boundaries to produce these chunks.

In [ ]:
ENTITY_COLORS = {
    "malware": "#ff6b6b",
    "vulnerability": "#ffa94d",
    "attack_technique": "#74c0fc",
    "affected_software": "#69db7c",
    "threat_actor": "#da77f2",
}

DEFAULT_ENTITY_TYPES = ["malware", "vulnerability", "attack_technique", "affected_software", "threat_actor"]


def run_ner(text: str, entity_types: list, threshold: float) -> list:
    """Run GLiNER and convert to HighlightedText format."""
    if not text.strip() or not entity_types:
        return [(text, None)]

    entities = ner_model.predict_entities(text, entity_types, threshold=threshold)
    # Sort by start position
    entities = sorted(entities, key=lambda e: e["start"])

    # Build (chunk, label) pairs
    result = []
    cursor = 0
    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]
        if start > cursor:
            result.append((text[cursor:start], None))
        result.append((text[start:end], label))
        cursor = end
    if cursor < len(text):
        result.append((text[cursor:], None))
    return result if result else [(text, None)]


# Test
sample = SAMPLE_TEXTS[0]['text']
result = run_ner(sample, DEFAULT_ENTITY_TYPES, threshold=0.4)
print(f"✓ NER result: {[(label, chunk[:30]) for chunk, label in result if label]}")

In [ ]:
with gr.Blocks(title="NER Demo") as ner_demo:
    gr.Markdown("# GLiNER — Named Entity Recognition\nZero-shot NER for cybersecurity texts.")

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Input text",
                lines=5,
                placeholder="Paste cybersecurity report here..."
            )
            entity_types = gr.CheckboxGroup(
                choices=DEFAULT_ENTITY_TYPES,
                value=DEFAULT_ENTITY_TYPES,
                label="Entity types"
            )
            threshold = gr.Slider(0.1, 0.9, value=0.4, step=0.05, label="Confidence threshold")
            run_btn = gr.Button("Extract entities", variant="primary")

        with gr.Column(scale=2):
            ner_output = gr.HighlightedText(
                label="Entities",
                color_map=ENTITY_COLORS,
                show_legend=True,
            )

    gr.Examples(
        examples=[[t['text']] for t in SAMPLE_TEXTS[:4]],
        inputs=text_input,
        label="Example reports"
    )

    run_btn.click(run_ner, inputs=[text_input, entity_types, threshold], outputs=ner_output)

ner_demo.launch()

## 3. Classification Demo with gr.Label

In [ ]:
ATTACK_LABELS = [
    "ransomware", "phishing", "apt_attack", "ddos",
    "data_breach", "supply_chain_attack", "zero_day"
]


def run_cls(text: str, labels_str: str) -> dict:
    """Run GLiClass and return label → score dict for gr.Label."""
    labels = [l.strip() for l in labels_str.split(',') if l.strip()]
    if not labels or not text.strip():
        return {}
    results = cls_pipeline(text, labels, threshold=0.0)[0]
    return {r['label']: round(float(r['score']), 4) for r in results}


# Test
scores = run_cls(SAMPLE_TEXTS[0]['text'], ', '.join(ATTACK_LABELS))
print(f"✓ Classification result: {scores}")

In [ ]:
with gr.Blocks(title="Classification Demo") as cls_demo:
    gr.Markdown("# GLiClass — Attack Type Classification\nZero-shot classification for cybersecurity reports.")

    with gr.Row():
        with gr.Column(scale=2):
            cls_text = gr.Textbox(label="Report text", lines=5)
            cls_labels = gr.Textbox(
                label="Labels (comma-separated)",
                value=", ".join(ATTACK_LABELS)
            )
            cls_btn = gr.Button("Classify", variant="primary")

        with gr.Column(scale=1):
            cls_output = gr.Label(label="Attack types", num_top_classes=7)

    gr.Examples(
        examples=[[t['text'], ', '.join(ATTACK_LABELS)] for t in SAMPLE_TEXTS[:4]],
        inputs=[cls_text, cls_labels],
        label="Examples"
    )

    cls_btn.click(run_cls, inputs=[cls_text, cls_labels], outputs=cls_output)

cls_demo.launch()

## 4. Combined Multi-tab App

In [ ]:
with gr.Blocks(title="Cybersecurity NLP Toolkit") as combined_demo:
    gr.Markdown("# Cybersecurity NLP Toolkit\nPowered by GLiNER + GLiClass edge models.")

    with gr.Tabs():
        # --- Tab 1: NER ---
        with gr.Tab("Named Entity Recognition"):
            with gr.Row():
                with gr.Column():
                    t1_text = gr.Textbox(label="Text", lines=5)
                    t1_types = gr.CheckboxGroup(
                        choices=DEFAULT_ENTITY_TYPES,
                        value=DEFAULT_ENTITY_TYPES,
                        label="Entity types"
                    )
                    t1_thresh = gr.Slider(0.1, 0.9, value=0.4, step=0.05, label="Threshold")
                    gr.Button("Extract", variant="primary").click(
                        run_ner, [t1_text, t1_types, t1_thresh],
                        gr.HighlightedText(label="Entities", color_map=ENTITY_COLORS, show_legend=True)
                    )
            gr.Examples([[t['text']] for t in SAMPLE_TEXTS[:3]], inputs=t1_text)

        # --- Tab 2: Classification ---
        with gr.Tab("Attack Classification"):
            with gr.Row():
                with gr.Column():
                    t2_text = gr.Textbox(label="Report text", lines=5)
                    t2_labels = gr.Textbox(
                        label="Labels (comma-separated)",
                        value=", ".join(ATTACK_LABELS)
                    )
                    gr.Button("Classify", variant="primary").click(
                        run_cls, [t2_text, t2_labels],
                        gr.Label(label="Scores", num_top_classes=7)
                    )
            gr.Examples([[t['text']] for t in SAMPLE_TEXTS[:3]], inputs=t2_text)

combined_demo.launch()